## 1.提示词工程
- 把SystemMessage称为系统提示词,即system_prompt
- 而提示词工程，就是通过优化系统提示词使模型输出的结果更符合业务需求。使用Markdown格式或者XML格式编写。     

1.身份（Identity）：描述AI的职责、沟通风格和总体目标。  
2.说明（Instructions）：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？  
3.示例（Examples）：提供可能的输入示例，以及模型期望的输出。===》为了让传统程序识别，我们要实例结构化输出（JSON）  
4.背景信息（Context）：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。

In [5]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

#提示词工程
system_prompt="""
# 身份
- 你是奶块游戏的开发的设计AI助手，根据用户的要求设计游戏武器。

#指令
- 请务必以JSON格式速出
- 不要加任何Markdown样式

#示例
user:你给我设计一把近战型的武器.
assistant:
{
"name": "龙枪",
"description": "这是一把近战武器，可以攻击敌人，并给自己加防护",
"attack": 10,
"defense": 5
}
"""

#初始化模型
clint=create_agent(
    "deepseek-chat",
    system_prompt=system_prompt
)

In [ ]:
#调用模型,流式
respone=clint.stream({
    "messages":[HumanMessage(content="你给我设计一把远程型的武器.")]
},
stream_mode="messages"
)

for chunk,metadata in respone:
    if chunk.content:
        print(chunk.content,end="",flush=True)

#chunk 是消息片段，，metadata 是后台附加信息

In [6]:
#调用模型，阻塞式
response=clint.invoke({
    "messages":[HumanMessage(content="你给我设计一个防御装备")]
})
print(response)

{'messages': [HumanMessage(content='你给我设计一个防御装备', additional_kwargs={}, response_metadata={}, id='9509354a-cafa-456d-aa21-5e9c5380782a'), AIMessage(content='{\n"name": "圣盾护甲",\n"description": "这是一件防御装备，可以大幅提升穿戴者的物理和魔法防御能力，并附带生命恢复效果",\n"defense": 25,\n"health": 50\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 111, 'total_tokens': 161, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 111}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '2917da0a-f0a4-47aa-9ae4-c4f0024fc8d7', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4a10-0047-7781-8c06-2a92624cdbf9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 50, 'total_tokens': 161, 'input_token_details': {'cach

In [ ]:
print(response['messages'][-1].content)

#response 是一个 Python 字典(dict)，不是对象
# 字典取数据 → 必须用 [] 中括号：字典名['键名']
#对象取属性 → 才用 . 点号：对象名.属性名

{
"name": "圣盾护甲",
"description": "这是一件防御装备，可以大幅提升穿戴者的物理和魔法防御能力，并附带生命恢复效果",
"defense": 25,
"health": 50
}


## 2.结构化输出（更推荐）
在langchain中，实现结构化输出会更加简单，我们无需在提示词中添加描述实现结构化输出，只需要设定一个数据类型即可。     
- 从Python库 pydantic 中，导入它的核心工具 BaseModel
- BaseModel 作用：你自己写的数据模型类，只要继承它，就能自动获得数据验证、类型检查、格式转换等超强能力
- response_format = .... (指定数据结构)

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel

class ResponseOut(BaseModel):
    name: str
    description: str
    defense:int
    attack:int

clint=create_agent(
    "deepseek-chat",
    system_prompt="你是奶块游戏的开发的设计AI助手，根据用户的要求设计游戏武器",
    response_format=ResponseOut
)
response=clint.invoke({
    "messages":[HumanMessage(content="请为游戏设计回血型武器")]
})

print(response['messages'])


{'messages': [HumanMessage(content='请为游戏设计回血型武器', additional_kwargs={}, response_metadata={}, id='6d022424-55af-4e1f-b0f4-fe954907521a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 334, 'total_tokens': 479, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 78}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '9fd679fb-e90b-4a9d-bbe4-f1634d72f4dc', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e4a27-06df-7d31-bbd2-6c3eafe2db85-0', tool_calls=[{'name': 'ResponseOut', 'args': {'name': '生命之泉·圣剑', 'description': '一把由纯净水晶与生命之泉淬炼而成的圣洁之剑，剑身流转着翠绿色的生命光晕。每次攻击可吸取目标生命值并转化为自身血量，同时有一定概率触发"生命绽放"效果，为周围友军回复少量生命值。', 'defense': 5, 'attack': 12}, 'id': 'call_00_klSTxlOKV38RrSbB3BTO5939', 'ty

In [13]:
print(response['structured_response'])

name='生命之泉·圣剑' description='一把由纯净水晶与生命之泉淬炼而成的圣洁之剑，剑身流转着翠绿色的生命光晕。每次攻击可吸取目标生命值并转化为自身血量，同时有一定概率触发"生命绽放"效果，为周围友军回复少量生命值。' defense=5 attack=12
